In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import shap
from sklearn.model_selection import KFold, cross_val_score
from sklearn.utils import shuffle
import joblib
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
import xgboost as xgb
from sklearn.base import clone
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

In [ ]:
try:
    # For Colab
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv('Composition_dataset_wt%.csv')
except:
    # For local execution
    df = pd.read_csv('../data/training/Composition_dataset_wt%.csv')

print(f"Loaded {len(df)} samples with {len(df.columns)} features")

In [ ]:
# ========================================
# CONFIGURATION
# ========================================
RUN_OPTIMIZATION = False  # True: run hyperparameter tuning | False: use pre-optimized parameters
USE_EXTERNAL_VALIDATION = False  # True: run external validation | False: skip validation
SAVE_MODEL = False  # True: save model to .pkl file | False: skip saving
SAVE_FOLDS = False  # Set to False to skip saving and downloading
SAVE_CV_RESULTS = True  # Set to False to skip saving
property_name = "Js"           # "Hc", "Js", or "rho"
featurization_method = "Composition"  # "Composition", "WenAlloys", or "Oliynyk"
# K-Fold cross-validation parameters
N_SPLITS = 10
RANDOM_STATE = 42
SAVE_GRIDSHAP_PLOT = False  # Set to False to skip downloading Grid-SHAP plots

# Optuna optimization settings (use None to skip parameter)
# Example:
OPTUNA_CONFIG = {
    'RFR': {'n_trials': 100, 'timeout': None},   # Only 100 trials
    'ETR': {'n_trials': None, 'timeout': 600},   # Only 10 min timeout
    'GBR': {'n_trials': 150, 'timeout': 900},    # Both: 150 trials or 15 min
    'XGB': {'n_trials': 150, 'timeout': 900}     # Both: 150 trials or 15 min
}

In [ ]:
if RUN_OPTIMIZATION:
    # For Google Colab: uncomment the line below to install Optuna
    !pip install -q optuna

    # For local use: install via requirements.txt
    # pip install -r requirements.txt

    import optuna
    import  optuna.visualization as vis

In [ ]:
if USE_EXTERNAL_VALIDATION:
    # File creation
    output_file = f"predictions_{property_name}_{featurization_method}.xlsx"
    column_name = f"{property_name}_predict"

    # Load experimental dataset
    try:
        # Colab: upload file
        uploaded = files.upload()
        X_predict = pd.read_csv('Dataset_experimental_wt%.csv')
    except:
        # Local: load from directory
        X_predict = pd.read_csv('../data/external_validation/Dataset_experimental_wt%.csv')

    print(f"Loaded {len(X_predict)} experimental samples with {len(X_predict.columns)} features")
    display(X_predict.head())

    # Initialize prediction results table (5 alloys × 2 models)
    results = pd.DataFrame({
        "Model": ["ETR"]*5 + ["XGB"]*5,
        "Alloy_number": list(range(1, 6)) + list(range(1, 6)),
        column_name: [np.nan]*10
    })

    print(f"\nPrediction table initialized for {property_name} ({featurization_method})")
    print(results, "\n")

In [ ]:
# Exploratory Data Analysis
df.describe()

In [ ]:
# Data Preprocessing
df_drop = df.dropna(subset=['Saturation polarization'])  # Remove rows with missing target values
df_drop = df_drop.reset_index(drop=True)     # Reset index after dropping

In [ ]:
# Feature and Target Extraction
features = df_drop.iloc[:, 0:3]              # Si, Al, Fe composition (wt.%)
target= df_drop.loc[:, 'Saturation polarization']       # Target: Saturation polarization (T)

In [ ]:
# Data Shuffling and Splitting
features, target = shuffle(features, target, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

In [ ]:
# ============================================================
# GENERATE & SAVE FOLDS
# ============================================================

# Reset index to ensure positional access works correctly
features_cv = features.reset_index(drop=True)
target_cv   = target.reset_index(drop=True)

# Generate folds once — reuse across all featurization notebooks
kf    = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
folds = list(kf.split(np.arange(len(target_cv))))

if SAVE_FOLDS:
    fold_train = np.array([f[0] for f in folds], dtype=object)
    fold_test  = np.array([f[1] for f in folds], dtype=object)
    np.save(f'folds_{property_name}_train.npy', fold_train, allow_pickle=True)
    np.save(f'folds_{property_name}_test.npy',  fold_test,  allow_pickle=True)
    print(f"✅ Folds saved: 'folds_{property_name}_train.npy', 'folds_{property_name}_test.npy'")

    try:
        files.download(f'folds_{property_name}_train.npy')
        files.download(f'folds_{property_name}_test.npy')
    except:
        pass

print(f"Generated {N_SPLITS} folds | Total samples: {len(target_cv)}")

# Optuna Hyperparameters Optimization

# Random Forest

In [ ]:
def objective(trial):
    """Optuna objective for Random Forest hyperparameter tuning."""

    # Hyperparameter search space
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 325),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 4),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 2),
    }

    # Evaluate with 5-fold CV
    model = RandomForestRegressor(random_state=42, **params)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()

    return score

In [ ]:
if RUN_OPTIMIZATION:
    print("Running hyperparameter optimization...")
    study = optuna.create_study(direction='maximize')

    # Extract config and filter out None values
    config = OPTUNA_CONFIG['RFR']
    optimize_kwargs = {k: v for k, v in config.items() if v is not None}

    study.optimize(objective, **optimize_kwargs)

    # Display results
    print('\nBest trial results:')
    trial = study.best_trial
    print(f'R2 score: {trial.value:.4f}\n')

    print('Best parameters:')
    print('best_params = {')
    for i, (key, value) in enumerate(trial.params.items()):
        if isinstance(value, float):
            print(f"    '{key}': {value},")
        elif isinstance(value, str):
            print(f"    '{key}': '{value}',")
        else:
            print(f"    '{key}': {value},")
    print('}\n')

    best_params = study.best_params

    # Display Optuna optimization visualizations
    vis.plot_optimization_history(study).show()
    vis.plot_param_importances(study).show()
    vis.plot_parallel_coordinate(study).show()

else:
    # Use pre-optimized parameters
    print("Using pre-optimized hyperparameters...")
    best_params = {
        'n_estimators': 270,
        'max_depth': 10,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'criterion': 'friedman_mse'
    }
    print(f"Parameters: {best_params}\n")

# Train model with best parameters
best_model = RandomForestRegressor(random_state=42, **best_params)
best_model.fit(X_train, y_train)

In [ ]:
# ============================================================
# FINAL MODEL + CROSS-VALIDATION
# ============================================================

cv_all_results = {}

final_model = RandomForestRegressor(random_state=42, **best_params)
final_model.fit(features_cv, target_cv)

fold_results = []
all_y_true   = []
all_y_pred   = []
fold_indices = []

print(f"\n{'='*65}")
print(f"CV | Target: {property_name} | Features: {featurization_method} | N={len(target_cv)}")
print(f"{'='*65}")
print(f"{'Fold':<6} {'Train':<8} {'Test':<7} {'R²_train':<12} {'R²_test':<12} {'MAE_test'}")
print(f"{'-'*65}")

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    X_train_f = features_cv.iloc[train_idx]
    X_test_f  = features_cv.iloc[test_idx]
    y_train_f = target_cv.iloc[train_idx]
    y_test_f  = target_cv.iloc[test_idx]

    model_f = clone(final_model)
    model_f.fit(X_train_f, y_train_f)

    y_pred_train = model_f.predict(X_train_f)
    y_pred_test  = model_f.predict(X_test_f)

    r2_tr  = r2_score(y_train_f, y_pred_train)
    r2_te  = r2_score(y_test_f,  y_pred_test)
    mae_te = mean_absolute_error(y_test_f, y_pred_test)

    fold_results.append({
        'Fold': fold, 'train_size': len(train_idx), 'test_size': len(test_idx),
        'R2_train': r2_tr, 'R2_test': r2_te, 'MAE_test': mae_te,
        'test_idx': test_idx
    })

    all_y_true.extend(y_test_f.values)
    all_y_pred.extend(y_pred_test)
    fold_indices.extend([fold] * len(test_idx))

    print(f"{fold:<6} {len(train_idx):<8} {len(test_idx):<7} "
          f"{r2_tr:<12.4f} {r2_te:<12.4f} {mae_te:.4f}")

cv_all_results['RFR'] = (fold_results, all_y_true, all_y_pred)

print(f"{'-'*65}")
r2_te_vals  = [r['R2_test']  for r in fold_results]
mae_te_vals = [r['MAE_test'] for r in fold_results]
print(f"{'':6} {'':8} {'':7} {'Mean':<12} {np.mean(r2_te_vals):<12.4f} {np.mean(mae_te_vals):.4f}")
print(f"{'':6} {'':8} {'':7} {'±Std':<12} {np.std(r2_te_vals):<12.4f} {np.std(mae_te_vals):.4f}")

oof_r2 = r2_score(all_y_true, all_y_pred)
print(f"{'':6} {'':8} {'':7} {'OOF R²':<12} {oof_r2:.4f}")
print(f"{'='*65}")

In [ ]:
if SAVE_MODEL:
    # Save trained model
    joblib.dump(final_model, '1_composition_SP_rfr.pkl')
    print("✅ Model saved as '1_composition_SP_rfr.pkl'")

    try:
        files.download('1_composition_SP_rfr.pkl')
    except:
        pass

# Extra Trees Regressor

In [ ]:
def objective(trial):
    """Optuna objective for Extra Trees hyperparameter tuning."""

    # Hyperparameter search space
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 180, 325),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 8),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 2),
        'criterion': trial.suggest_categorical('criterion', ['squared_error', 'absolute_error', 'friedman_mse', 'poisson']),
            }

    # Evaluate with 5-fold CV
    model = ExtraTreesRegressor(random_state=42, **params)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()
    return score

In [ ]:
if RUN_OPTIMIZATION:
    print("Running hyperparameter optimization...")
    study = optuna.create_study(direction='maximize')

    # Extract config and filter out None values
    config = OPTUNA_CONFIG['ETR']
    optimize_kwargs = {k: v for k, v in config.items() if v is not None}

    study.optimize(objective, **optimize_kwargs)

    # Display results
    print('\nBest trial results:')
    trial = study.best_trial
    print(f'R2 score: {trial.value:.4f}\n')

    print('Best parameters:')
    print('best_params = {')
    for i, (key, value) in enumerate(trial.params.items()):
        if isinstance(value, float):
            print(f"    '{key}': {value},")
        elif isinstance(value, str):
            print(f"    '{key}': '{value}',")
        else:
            print(f"    '{key}': {value},")
    print('}\n')

    best_params = study.best_params

    # Display Optuna optimization visualizations
    vis.plot_optimization_history(study).show()
    vis.plot_param_importances(study).show()
    vis.plot_parallel_coordinate(study).show()

else:
    # Use pre-optimized parameters
    print("Using pre-optimized hyperparameters...")
    best_params = {
        'n_estimators': 270,
        'max_depth': 7,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'criterion': 'poisson'
    }
    print(f"Parameters: {best_params}\n")

# Train model with best parameters
best_model = ExtraTreesRegressor(random_state=42, **best_params)
best_model.fit(X_train, y_train)

In [ ]:
# ============================================================
# FINAL MODEL + CROSS-VALIDATION
# ============================================================

final_model = ExtraTreesRegressor(random_state=42, **best_params)
final_model.fit(features_cv, target_cv)

fold_results = []
all_y_true   = []
all_y_pred   = []
fold_indices = []

print(f"\n{'='*65}")
print(f"CV | Target: {property_name} | Features: {featurization_method} | N={len(target_cv)}")
print(f"{'='*65}")
print(f"{'Fold':<6} {'Train':<8} {'Test':<7} {'R²_train':<12} {'R²_test':<12} {'MAE_test'}")
print(f"{'-'*65}")

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    X_train_f = features_cv.iloc[train_idx]
    X_test_f  = features_cv.iloc[test_idx]
    y_train_f = target_cv.iloc[train_idx]
    y_test_f  = target_cv.iloc[test_idx]

    model_f = clone(final_model)
    model_f.fit(X_train_f, y_train_f)

    y_pred_train = model_f.predict(X_train_f)
    y_pred_test  = model_f.predict(X_test_f)

    r2_tr  = r2_score(y_train_f, y_pred_train)
    r2_te  = r2_score(y_test_f,  y_pred_test)
    mae_te = mean_absolute_error(y_test_f, y_pred_test)

    fold_results.append({
        'Fold': fold, 'train_size': len(train_idx), 'test_size': len(test_idx),
        'R2_train': r2_tr, 'R2_test': r2_te, 'MAE_test': mae_te,
        'test_idx': test_idx
    })

    all_y_true.extend(y_test_f.values)
    all_y_pred.extend(y_pred_test)
    fold_indices.extend([fold] * len(test_idx))

    print(f"{fold:<6} {len(train_idx):<8} {len(test_idx):<7} "
          f"{r2_tr:<12.4f} {r2_te:<12.4f} {mae_te:.4f}")

cv_all_results['ETR'] = (fold_results, all_y_true, all_y_pred)

print(f"{'-'*65}")
r2_te_vals  = [r['R2_test']  for r in fold_results]
mae_te_vals = [r['MAE_test'] for r in fold_results]
print(f"{'':6} {'':8} {'':7} {'Mean':<12} {np.mean(r2_te_vals):<12.4f} {np.mean(mae_te_vals):.4f}")
print(f"{'':6} {'':8} {'':7} {'±Std':<12} {np.std(r2_te_vals):<12.4f} {np.std(mae_te_vals):.4f}")

oof_r2 = r2_score(all_y_true, all_y_pred)
print(f"{'':6} {'':8} {'':7} {'OOF R²':<12} {oof_r2:.4f}")
print(f"{'='*65}")

In [ ]:
if SAVE_MODEL:

# Save model to .pkl file
    joblib.dump(final_model, '2_composition_SP_etr.pkl')

    print("✅ Model saved as '2_composition_SP_etr.pkl'")
    try:
        files.download('2_composition_SP_etr.pkl')
    except:
        pass

In [ ]:
if USE_EXTERNAL_VALIDATION:
    # Make predictions on external validation set (ETR)
    y_pred_etr = final_model.predict(X_predict)

    # Save ETR predictions to results DataFrame
    results.loc[results["Model"] == "ETR", column_name] = y_pred_etr
    print(f"ETR predictions saved: {y_pred_etr}")
    print(results)
    print("\n")

In [ ]:
def shifted_coolwarm_cmap(p, n=256):
    """Shift coolwarm colormap so position p becomes neutral center."""
    base = plt.get_cmap("coolwarm")
    xs = np.linspace(0, 1, n)

    cols = []
    for x in xs:
        if x <= p:
            t = 0.5 * (x / p) if p > 0 else 0.0
        else:
            t = 0.5 + 0.5 * ((x - p) / (1 - p)) if p < 1 else 1.0
        cols.append(base(t))

    return mcolors.ListedColormap(cols, name="coolwarm_shifted")



def make_coolwarm_saturated_with_center(values, vcenter=0.0, nbins=6, steps=(1, 2, 2.5, 5, 10)):
    """Create norm, cmap, and ticks for SHAP with saturated colors and zero at neutral."""
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]

    # Fallback for empty data
    if x.size == 0:
        norm = mcolors.Normalize(vmin=-1, vmax=1, clip=True)
        cmap = plt.get_cmap("coolwarm")
        ticks = np.array([-1, 0, 1], dtype=float)
        return norm, cmap, ticks

    data_min, data_max = float(np.min(x)), float(np.max(x))

    # Normalize by actual data range for saturated colors
    norm = mcolors.Normalize(vmin=data_min, vmax=data_max, clip=True)

    # Generate scientific ticks
    loc = mticker.MaxNLocator(nbins=nbins, steps=list(steps))
    ticks = loc.tick_values(data_min, data_max)
    ticks = ticks[(ticks >= norm.vmin) & (ticks <= norm.vmax)]

    # Shift colormap to place vcenter at neutral
    if data_min < vcenter < data_max:
        p = (vcenter - data_min) / (data_max - data_min)
        cmap = shifted_coolwarm_cmap(p, n=256)

        # Ensure vcenter is in ticks
        if not np.any(np.isclose(ticks, vcenter)):
            ticks = np.sort(np.append(ticks, vcenter))
    else:
        cmap = plt.get_cmap("coolwarm")

    return norm, cmap, ticks

In [ ]:
# ============ CALCULATE SHAP VALUES ============
explainer   = shap.TreeExplainer(final_model, features_cv)
shap_values = explainer(features_cv)

# Create Explanation object (used for beeswarm below)
explanation = shap.Explanation(
    values=shap_values.values,
    base_values=explainer.expected_value,
    data=features_cv.values,
    feature_names=features_cv.columns.tolist()
)

# ============ DEFINE FEATURE ORDER ============
desired_order = ['Fe', 'Si', 'Al']

# ============ BUILD GRID-SHAP PLOTS ============
plt.style.use('default')
fig = plt.figure(figsize=(3.5, 10))
gs = GridSpec(3, 1, figure=fig, hspace=0.3)

for i, feature_name in enumerate(desired_order):
    try:
        feature_index = features_cv.columns.get_loc(feature_name)
    except KeyError:
        print(f"Error: Feature '{feature_name}' not found in features_cv.")
        continue

    ax = fig.add_subplot(gs[i, 0])
    ax.set_facecolor('#f0f0f0')

    shap_col = shap_values[:, feature_index].values

    norm, cmap_local, ticks = make_coolwarm_saturated_with_center(
        shap_col,
        vcenter=0.0,
        nbins=6
    )

    sc = ax.scatter(
        features_cv[feature_name],
        target_cv,
        c=shap_col,
        cmap=cmap_local,
        norm=norm,
        alpha=0.9,
        s=50,
        edgecolor='black',
        linewidth=0.3,
        marker='o'
    )

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='4.5%', pad=0.05)
    cbar = fig.colorbar(sc, cax=cax)

    cbar.set_ticks(ticks)
    cbar.formatter = mticker.ScalarFormatter(useMathText=False)
    cbar.formatter.set_powerlimits((-100, 100))
    cbar.update_ticks()
    cbar.ax.tick_params(labelsize=11)
    cbar.set_label('SHAP Value', fontsize=12, labelpad=5)

    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=5))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(min_n_ticks=3, nbins=4, integer=True))
    ax.set_xlabel(feature_name, fontsize=12, labelpad=4)
    ax.set_ylabel("Saturation Polarization (T)", fontsize=12, labelpad=4)
    ax.tick_params(axis='both', labelsize=11, pad=4)
    ax.grid(True, linestyle=':', alpha=0.35, linewidth=0.45)

gridshap_filename = f'{featurization_method}_Grid-SHAP_{property_name}_ETR.png'
plt.savefig(gridshap_filename, dpi=600, bbox_inches='tight', facecolor='white')
plt.show()

if SAVE_GRIDSHAP_PLOT:
    try:
        files.download(gridshap_filename)
    except:
        pass

# Gradient Boost Regressor

In [ ]:
def objective(trial):
    """Optuna objective for Gradient Boost hyperparameter tuning."""

    # Hyperparameter search space
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 180, 350),
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 2, 7),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 6),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'subsample': trial.suggest_float('subsample', 0.6, 0.8),
        'loss': 'squared_error',
    }

    # Evaluate with 5-fold CV
    model = GradientBoostingRegressor(random_state=42, **params)
    score = cross_val_score(model, X_train, y_train,
                          cv=5, scoring='r2').mean()

    return score

In [ ]:
if RUN_OPTIMIZATION:
    print("Running hyperparameter optimization...")
    study = optuna.create_study(direction='maximize')

    # Extract config and filter out None values
    config = OPTUNA_CONFIG['GBR']
    optimize_kwargs = {k: v for k, v in config.items() if v is not None}

    study.optimize(objective, **optimize_kwargs)

    # Display results
    print('\nBest trial results:')
    trial = study.best_trial
    print(f'R2 score: {trial.value:.4f}\n')

    print('Best parameters:')
    print('best_params = {')
    for i, (key, value) in enumerate(trial.params.items()):
        if isinstance(value, float):
            print(f"    '{key}': {value},")
        elif isinstance(value, str):
            print(f"    '{key}': '{value}',")
        else:
            print(f"    '{key}': {value},")
    print('}\n')

    best_params = study.best_params

    # Display Optuna optimization visualizations
    vis.plot_optimization_history(study).show()
    vis.plot_param_importances(study).show()
    vis.plot_parallel_coordinate(study).show()

else:
    # Use pre-optimized parameters
    print("Using pre-optimized hyperparameters...")
    best_params = {
        'n_estimators': 180,
        'learning_rate': 0.08,
        'max_depth': 3,
        'min_samples_split': 6,
        'min_samples_leaf': 3,
        'subsample': 0.64,
        'loss': 'squared_error'
    }
    print(f"Parameters: {best_params}\n")

# Train model with best parameters
best_model = GradientBoostingRegressor(random_state=42, **best_params)
best_model.fit(X_train, y_train)

In [ ]:
# ============================================================
# FINAL MODEL + CROSS-VALIDATION
# ============================================================

final_model = GradientBoostingRegressor(random_state=42, **best_params)
final_model.fit(features_cv, target_cv)

fold_results = []
all_y_true   = []
all_y_pred   = []
fold_indices = []

print(f"\n{'='*65}")
print(f"CV | Target: {property_name} | Features: {featurization_method} | N={len(target_cv)}")
print(f"{'='*65}")
print(f"{'Fold':<6} {'Train':<8} {'Test':<7} {'R²_train':<12} {'R²_test':<12} {'MAE_test'}")
print(f"{'-'*65}")

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    X_train_f = features_cv.iloc[train_idx]
    X_test_f  = features_cv.iloc[test_idx]
    y_train_f = target_cv.iloc[train_idx]
    y_test_f  = target_cv.iloc[test_idx]

    model_f = clone(final_model)
    model_f.fit(X_train_f, y_train_f)

    y_pred_train = model_f.predict(X_train_f)
    y_pred_test  = model_f.predict(X_test_f)

    r2_tr  = r2_score(y_train_f, y_pred_train)
    r2_te  = r2_score(y_test_f,  y_pred_test)
    mae_te = mean_absolute_error(y_test_f, y_pred_test)

    fold_results.append({
        'Fold': fold, 'train_size': len(train_idx), 'test_size': len(test_idx),
        'R2_train': r2_tr, 'R2_test': r2_te, 'MAE_test': mae_te,
        'test_idx': test_idx
    })

    all_y_true.extend(y_test_f.values)
    all_y_pred.extend(y_pred_test)
    fold_indices.extend([fold] * len(test_idx))

    print(f"{fold:<6} {len(train_idx):<8} {len(test_idx):<7} "
          f"{r2_tr:<12.4f} {r2_te:<12.4f} {mae_te:.4f}")

cv_all_results['GBR'] = (fold_results, all_y_true, all_y_pred)

print(f"{'-'*65}")
r2_te_vals  = [r['R2_test']  for r in fold_results]
mae_te_vals = [r['MAE_test'] for r in fold_results]
print(f"{'':6} {'':8} {'':7} {'Mean':<12} {np.mean(r2_te_vals):<12.4f} {np.mean(mae_te_vals):.4f}")
print(f"{'':6} {'':8} {'':7} {'±Std':<12} {np.std(r2_te_vals):<12.4f} {np.std(mae_te_vals):.4f}")

oof_r2 = r2_score(all_y_true, all_y_pred)
print(f"{'':6} {'':8} {'':7} {'OOF R²':<12} {oof_r2:.4f}")
print(f"{'='*65}")

In [ ]:
if SAVE_MODEL:

# Save model to .pkl file
    joblib.dump(final_model, '3_composition_SP_gbr.pkl')

    print("✅ Model saved as '3_composition_SP_gbr.pkl'")
    try:
        files.download('3_composition_SP_gbr.pkl')
    except:
        pass

# XGBoost Regressor

In [ ]:
def objective(trial):
    """Optuna objective for Gradient Boost hyperparameter tuning."""

    # Hyperparameter search space
    params = {
        'objective': 'reg:squarederror',
        'n_estimators': trial.suggest_int('n_estimators', 350, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.1, 0.25, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 0.95),
        'min_child_weight': 1,
        'gamma': trial.suggest_float('gamma', 0, 0.1),
        'alpha': trial.suggest_float('alpha', 0, 2),
    }

    # Evaluate with 5-fold CV
    model = xgb.XGBRegressor(random_state=42, **params)
    score = cross_val_score(model, X_train, y_train,
                            cv=5, scoring='r2').mean()

    return score

In [ ]:
if RUN_OPTIMIZATION:
    print("Running hyperparameter optimization...")
    study = optuna.create_study(direction='maximize')

    # Extract config and filter out None values
    config = OPTUNA_CONFIG['XGB']
    optimize_kwargs = {k: v for k, v in config.items() if v is not None}

    study.optimize(objective, **optimize_kwargs)

    # Display results
    print('\nBest trial results:')
    trial = study.best_trial
    print(f'R2 score: {trial.value:.4f}\n')

    print('Best parameters:')
    print('best_params = {')
    for i, (key, value) in enumerate(trial.params.items()):
        if isinstance(value, float):
            print(f"    '{key}': {value},")
        elif isinstance(value, str):
            print(f"    '{key}': '{value}',")
        else:
            print(f"    '{key}': {value},")
    print('}\n')

    best_params = study.best_params

    # Display Optuna optimization visualizations
    vis.plot_optimization_history(study).show()
    vis.plot_param_importances(study).show()
    vis.plot_parallel_coordinate(study).show()

else:
    # Use pre-optimized parameters
    print("Using pre-optimized hyperparameters...")
    best_params = {
        'n_estimators': 350,
        'max_depth': 3,
        'learning_rate': 0.15,
        'subsample': 0.74,
        'colsample_bytree': 0.88,
        'min_child_weight': 1,
        'gamma': 0.01,
        'alpha': 0.01
    }
    print(f"Parameters: {best_params}\n")

# Train model with best parameters
mean_target = float(y_train.mean())
best_model = xgb.XGBRegressor(random_state=42, **best_params)
best_model.fit(X_train, y_train)

In [ ]:
# ============================================================
# FINAL MODEL + CROSS-VALIDATION
# ============================================================

final_model = xgb.XGBRegressor(random_state=42, **best_params)
final_model.fit(features_cv, target_cv)

fold_results = []
all_y_true   = []
all_y_pred   = []
fold_indices = []

print(f"\n{'='*65}")
print(f"CV | Target: {property_name} | Features: {featurization_method} | N={len(target_cv)}")
print(f"{'='*65}")
print(f"{'Fold':<6} {'Train':<8} {'Test':<7} {'R²_train':<12} {'R²_test':<12} {'MAE_test'}")
print(f"{'-'*65}")

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    X_train_f = features_cv.iloc[train_idx]
    X_test_f  = features_cv.iloc[test_idx]
    y_train_f = target_cv.iloc[train_idx]
    y_test_f  = target_cv.iloc[test_idx]

    model_f = clone(final_model)
    model_f.fit(X_train_f, y_train_f)

    y_pred_train = model_f.predict(X_train_f)
    y_pred_test  = model_f.predict(X_test_f)

    r2_tr  = r2_score(y_train_f, y_pred_train)
    r2_te  = r2_score(y_test_f,  y_pred_test)
    mae_te = mean_absolute_error(y_test_f, y_pred_test)

    fold_results.append({
        'Fold': fold, 'train_size': len(train_idx), 'test_size': len(test_idx),
        'R2_train': r2_tr, 'R2_test': r2_te, 'MAE_test': mae_te,
        'test_idx': test_idx
    })

    all_y_true.extend(y_test_f.values)
    all_y_pred.extend(y_pred_test)
    fold_indices.extend([fold] * len(test_idx))

    print(f"{fold:<6} {len(train_idx):<8} {len(test_idx):<7} "
          f"{r2_tr:<12.4f} {r2_te:<12.4f} {mae_te:.4f}")

cv_all_results['XGB'] = (fold_results, all_y_true, all_y_pred)

print(f"{'-'*65}")
r2_te_vals  = [r['R2_test']  for r in fold_results]
mae_te_vals = [r['MAE_test'] for r in fold_results]
print(f"{'':6} {'':8} {'':7} {'Mean':<12} {np.mean(r2_te_vals):<12.4f} {np.mean(mae_te_vals):.4f}")
print(f"{'':6} {'':8} {'':7} {'±Std':<12} {np.std(r2_te_vals):<12.4f} {np.std(mae_te_vals):.4f}")

oof_r2 = r2_score(all_y_true, all_y_pred)
print(f"{'':6} {'':8} {'':7} {'OOF R²':<12} {oof_r2:.4f}")
print(f"{'='*65}")

In [ ]:
if SAVE_MODEL:

# Save model to .pkl file
    joblib.dump(final_model, '4_composition_SP_xgb.pkl')

    print("✅ Model saved as '4_composition_SP_xgb.pkl'")
    try:
        files.download('4_composition_SP_xgb.pkl')
    except:
        pass

In [ ]:
if USE_EXTERNAL_VALIDATION:
    # Make predictions on external validation set
    y_pred_xgb = final_model.predict(X_predict)

    # Save predictions to results DataFrame
    results.loc[results["Model"] == "XGB", column_name] = y_pred_xgb
    print(f"XGB predictions saved: {y_pred_xgb}")
    print(results)
    print("\n")

    # Add experimental ground truth values
    y_true = [0.937, 0.884, 0.607, 0.794, 1.574]  # ← UPDATE WITH ACTUAL VALUES
    results[f"{property_name}_true"] = y_true * 2  # Duplicate for both models

    # Calculate per-sample errors
    results["Residual"]  = results[column_name] - results[f"{property_name}_true"]
    results["AE"]        = results["Residual"].abs()                                      # Absolute Error
    results["APE (%)"]   = 100 * results["AE"] / results[f"{property_name}_true"]        # Absolute Percentage Error

    # Calculate average metrics per model (MAE and MAPE)
    for model in ["ETR", "XGB"]:
        mask = results["Model"] == model
        mae  = results.loc[mask, "AE"].mean()
        mape = results.loc[mask, "APE (%)"].mean()
        print(f"{model}: MAE = {mae:.4f}, MAPE = {mape:.2f}%")

    # Save results to Excel
    results.to_excel(output_file, index=False, engine='openpyxl')
    print(f"✓ Results saved to {output_file}")

    # Download file (Colab only)
    try:
        files.download(output_file)
    except:
        pass

In [ ]:
# ============ FUNCTION 1: Shifted coolwarm ============
def shifted_coolwarm_cmap(p, n=256):
    """
    Shift coolwarm colormap so position p becomes neutral center.

    """
    base = plt.get_cmap("coolwarm")
    xs = np.linspace(0, 1, n)

    cols = []
    for x in xs:
        if x <= p:
            t = 0.5 * (x / p) if p > 0 else 0.0
        else:
            t = 0.5 + 0.5 * ((x - p) / (1 - p)) if p < 1 else 1.0
        cols.append(base(t))

    return mcolors.ListedColormap(cols, name="coolwarm_shifted")


# ============ FUNCTION 2: Normalization + ticks + colormap ============
def make_coolwarm_saturated_with_center(values, vcenter=0.0, nbins=6, steps=(1, 2, 2.5, 5, 10)):
    """
    Create norm, cmap, and ticks for SHAP visualization with saturated colors and zero at neutral.

    """
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]

    # Fallback for empty data
    if x.size == 0:
        norm = mcolors.Normalize(vmin=-1, vmax=1, clip=True)
        cmap = plt.get_cmap("coolwarm")
        ticks = np.array([-1, 0, 1], dtype=float)
        return norm, cmap, ticks

    data_min, data_max = float(np.min(x)), float(np.max(x))

    # Normalize by actual data range for saturated colors
    norm = mcolors.Normalize(vmin=data_min, vmax=data_max, clip=True)

    # Generate scientific ticks
    loc = mticker.MaxNLocator(nbins=nbins, steps=list(steps))
    ticks = loc.tick_values(data_min, data_max)

    # Clip ticks to norm range
    ticks = ticks[(ticks >= norm.vmin) & (ticks <= norm.vmax)]

    # Shift colormap to place vcenter at neutral
    if data_min < vcenter < data_max:
        p = (vcenter - data_min) / (data_max - data_min)
        cmap = shifted_coolwarm_cmap(p, n=256)

        # Ensure vcenter is in ticks
        if not np.any(np.isclose(ticks, vcenter)):
            ticks = np.sort(np.append(ticks, vcenter))
    else:
        # Use standard coolwarm if vcenter is outside range
        cmap = plt.get_cmap("coolwarm")

    return norm, cmap, ticks

In [ ]:
# ============ EXTRACT SHAP FROM XGBOOST (DMatrix) ============
dmatrix_cv = xgb.DMatrix(features_cv, feature_names=features_cv.columns.tolist())

shap_values_raw    = final_model.get_booster().predict(dmatrix_cv, pred_contribs=True)
shap_values_matrix = shap_values_raw[:, :-1]
expected_value     = float(shap_values_raw[0, -1])

explanation = shap.Explanation(
    values=shap_values_matrix,
    base_values=np.full(len(features_cv), expected_value),
    data=features_cv.values,
    feature_names=features_cv.columns.tolist()
)

print(f"Shape SHAP values: {shap_values_matrix.shape}")
print(f"Expected value: {expected_value}")

# ============ DEFINE FEATURE ORDER ============
desired_order = ['Fe', 'Si', 'Al']

# ============ BUILD GRID-SHAP PLOTS ============
plt.style.use('default')
fig = plt.figure(figsize=(3.5, 10))
gs = GridSpec(3, 1, figure=fig, hspace=0.3)

for i, feature_name in enumerate(desired_order):
    try:
        feature_index = features_cv.columns.get_loc(feature_name)
    except KeyError:
        print(f"Error: Feature '{feature_name}' not found in features_cv.")
        continue

    ax = fig.add_subplot(gs[i, 0])
    ax.set_facecolor('#f0f0f0')

    shap_col = explanation[:, feature_index].values

    norm, cmap_local, ticks = make_coolwarm_saturated_with_center(
        shap_col,
        vcenter=0.0,
        nbins=6
    )

    sc = ax.scatter(
        features_cv[feature_name],
        target_cv,
        c=shap_col,
        cmap=cmap_local,
        norm=norm,
        alpha=0.9,
        s=50,
        edgecolor='black',
        linewidth=0.3,
        marker='o'
    )

    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='4.5%', pad=0.05)
    cbar = fig.colorbar(sc, cax=cax)

    cbar.set_ticks(ticks)
    cbar.formatter = mticker.ScalarFormatter(useMathText=False)
    cbar.formatter.set_powerlimits((-100, 100))
    cbar.update_ticks()
    cbar.ax.tick_params(labelsize=11)
    cbar.set_label('SHAP Value', fontsize=12, labelpad=5)

    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=5))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
    ax.set_xlabel(feature_name, fontsize=12, labelpad=4)
    ax.set_ylabel("Saturation Polarization (T)", fontsize=12, labelpad=4)
    ax.tick_params(axis='both', labelsize=11, pad=4)
    ax.grid(True, linestyle=':', alpha=0.35, linewidth=0.45)

gridshap_filename = f'{featurization_method}_Grid-SHAP_{property_name}_XGB.png'
plt.savefig(gridshap_filename, dpi=600, bbox_inches='tight', facecolor='white')
plt.show()

if SAVE_GRIDSHAP_PLOT:
    try:
        files.download(gridshap_filename)
    except:
        pass

# **Results**

In [ ]:
def build_cv_dataframe(fold_results, all_y_true, all_y_pred):
    """Convert fold_results list into a DataFrame with Mean, Std and OOF R² rows."""
    rows = []

    for r in fold_results:
        rows.append({
            'Fold':       r['Fold'],
            'Train size': r['train_size'],
            'Test size':  r['test_size'],
            'R² train':   round(r['R2_train'], 4),
            'R² test':    round(r['R2_test'],  4),
            'MAE test':   round(r['MAE_test'],  4),
        })

    r2_tr_vals  = [r['R2_train'] for r in fold_results]
    r2_te_vals  = [r['R2_test']  for r in fold_results]
    mae_te_vals = [r['MAE_test'] for r in fold_results]

    rows.append({
        'Fold':       '',
        'Train size': '',
        'Test size':  '',
        'R² train':   'Mean',
        'R² test':    round(np.mean(r2_te_vals),  4),
        'MAE test':   round(np.mean(mae_te_vals), 4),
    })

    rows.append({
        'Fold':       '',
        'Train size': '',
        'Test size':  '',
        'R² train':   '±Std',
        'R² test':    round(np.std(r2_te_vals),  4),
        'MAE test':   round(np.std(mae_te_vals), 4),
    })

    # OOF R² row
    rows.append({
        'Fold':       '',
        'Train size': '',
        'Test size':  '',
        'R² train':   'OOF R²',
        'R² test':    round(r2_score(all_y_true, all_y_pred), 4),
        'MAE test':   '',
    })

    return pd.DataFrame(rows)


def save_cv_to_xlsx(cv_all_results, property_name, featurization_method, filename=None):
    """Save CV results for all models to a single .xlsx with one sheet per model."""
    if filename is None:
        filename = f'cv_results_{property_name}_{featurization_method}.xlsx'

    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        for model_name, (fold_results, all_y_true, all_y_pred) in cv_all_results.items():
            df = build_cv_dataframe(fold_results, all_y_true, all_y_pred)
            df.to_excel(writer, sheet_name=model_name, index=False)

    wb = load_workbook(filename)
    header_fill = PatternFill('solid', fgColor='2E4057')
    mean_fill   = PatternFill('solid', fgColor='D9E8F5')
    std_fill    = PatternFill('solid', fgColor='EAF4FB')
    oof_fill    = PatternFill('solid', fgColor='D6EAD6')

    header_font = Font(bold=True, color='FFFFFF')
    bold_font   = Font(bold=True)
    center_align = Alignment(horizontal='center')

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]

        # Auto column width
        for col in ws.columns:
            max_len = max(len(str(cell.value)) if cell.value else 0 for cell in col)
            ws.column_dimensions[get_column_letter(col[0].column)].width = max_len + 4

        for row in ws.iter_rows():
            for cell in row:
                cell.alignment = center_align


                if cell.row == 1:
                    cell.fill = header_fill
                    cell.font = header_font
                    continue


                row_label = str(ws.cell(row=cell.row, column=4).value)

                if row_label in ['Mean', '±Std', 'OOF R²']:

                    if cell.column >= 4:
                        if row_label == 'Mean':
                            cell.fill = mean_fill
                            if cell.column in [4, 5, 6]: cell.font = bold_font
                        elif row_label == '±Std':
                            cell.fill = std_fill
                        elif row_label == 'OOF R²':
                            cell.fill = oof_fill
                            if cell.column in [4, 5, 6]: cell.font = bold_font

        # Add sheet title above the table
        ws.insert_rows(1)
        ws.cell(row=1, column=1,
                value=f'{sheet_name} | Property: {property_name} | Features: {featurization_method}')
        ws.cell(row=1, column=1).font = Font(bold=True, size=12)
        ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=6)

    wb.save(filename)
    print(f"✅ Saved: '{filename}'")
    return filename


# ============================================================
# SAVE
# ============================================================
if SAVE_CV_RESULTS:
    fname = save_cv_to_xlsx(cv_all_results, property_name, featurization_method)

    try:
        files.download(fname)
    except:
        pass